# 🧠 Medical Image Segmentation - Model Predictions
Simple notebook to load trained model and display predictions

In [ ]:
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============ SETTINGS ============
DEVICE = torch.device('cpu')
NUM_CLASSES = 55
MODEL_TYPE = 'meta'  # Change to: 'meta', 'proto', 'siamese', or 'fewshot_lite'

print(f"Model Type: {MODEL_TYPE}")
print(f"Device: {DEVICE}")
print(f"Classes: {NUM_CLASSES}")

In [ ]:
# ============ LOAD DATA ============
print("📁 Loading data...\n")

# Load images
image_dir = Path('train-images')
available_images = sorted([int(p.stem) for p in image_dir.glob('*.png')])
print(f"Found {len(available_images)} images")

images = []
for idx in tqdm(available_images, desc='Loading images'):
    img_path = image_dir / f'{idx}.png'
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        img = (img / 255.0).astype(np.float32)  # Normalize to [0, 1]
        images.append(img)

images = np.array(images)
print(f"Images shape: {images.shape}\n")

# Load labels
labels_df = pd.read_csv('y_train.csv', index_col=0)
labels_df = labels_df.T

# Match labels to images
labels = labels_df.iloc[:len(images)].values
labels_2d = labels.reshape(-1, 256, 256)
print(f"Labels shape: {labels_2d.shape}")
print(f"\n✅ Data loaded and aligned!")

In [ ]:
# ============ TRAIN SIMPLE META-UNET ============
print("\n🚀 Training Meta-U-Net...\n")

# Convert to torch tensors
images_tensor = torch.from_numpy(images[:, np.newaxis, :, :]).float()  # Add channel dim
labels_tensor = torch.from_numpy(labels_2d).long()

print(f"Images tensor shape: {images_tensor.shape}")
print(f"Labels tensor shape: {labels_tensor.shape}")

# Simple train/val split
n_train = int(0.8 * len(images_tensor))
train_indices = np.arange(n_train)
val_indices = np.arange(n_train, len(images_tensor))

X_train = images_tensor[train_indices]
y_train = labels_tensor[train_indices]
X_val = images_tensor[val_indices]
y_val = labels_tensor[val_indices]

print(f"\nTrain set: {X_train.shape}, Val set: {X_val.shape}")

In [ ]:
# ============ DEFINE META-UNET ============

class MetaUNet(torch.nn.Module):
    def __init__(self, in_channels=1, out_channels=55, base_channels=32):
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels, base_channels, 3, padding=1),
            torch.nn.BatchNorm2d(base_channels),
            torch.nn.ReLU(inplace=True),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(base_channels, base_channels*2, 3, padding=1),
            torch.nn.BatchNorm2d(base_channels*2),
            torch.nn.ReLU(inplace=True),
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Conv2d(base_channels*2, base_channels, 3, padding=1),
            torch.nn.BatchNorm2d(base_channels),
            torch.nn.ReLU(inplace=True),
            torch.nn.Upsample(scale_factor=2, mode='nearest'),
            torch.nn.Conv2d(base_channels, out_channels, 3, padding=1),
        )
    
    def forward(self, x):
        enc = self.encoder(x)
        dec = self.decoder(enc)
        return dec

model = MetaUNet(in_channels=1, out_channels=NUM_CLASSES)
model = model.to(DEVICE)
print(f"✅ Model created")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# ============ TRAINING LOOP ============
print("\n🎓 Training for 3 epochs...\n")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(3):
    # Train
    model.train()
    train_loss = 0
    
    # Mini-batch training
    batch_size = 2
    for i in range(0, len(X_train), batch_size):
        batch_x = X_train[i:i+batch_size].to(DEVICE)
        batch_y = y_train[i:i+batch_size].to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for i in range(0, len(X_val), batch_size):
            batch_x = X_val[i:i+batch_size].to(DEVICE)
            batch_y = y_val[i:i+batch_size].to(DEVICE)
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            val_loss += loss.item()
    
    train_loss /= len(X_train)
    val_loss /= len(X_val)
    
    print(f"Epoch {epoch+1}/3 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("\n✅ Training complete!")

In [ ]:
# ============ VISUALIZATION: IMAGE + GT + PREDICTION ============

def show_prediction(model, image_idx, images_tensor, labels_2d):
    """Display image, ground truth, and prediction side-by-side"""
    
    model.eval()
    with torch.no_grad():
        # Get image and label
        img_tensor = images_tensor[image_idx:image_idx+1].to(DEVICE)
        img_np = images_tensor[image_idx, 0].numpy()
        gt = labels_2d[image_idx]
        
        # Get prediction
        logits = model(img_tensor)
        pred = torch.argmax(logits, dim=1)[0].cpu().numpy()
        
        # Create figure
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # Original image
        axes[0].imshow(img_np, cmap='gray')
        axes[0].set_title('Original Image', fontsize=13, fontweight='bold')
        axes[0].axis('off')
        
        # Ground Truth
        im1 = axes[1].imshow(gt, cmap='tab20', interpolation='nearest')
        axes[1].set_title('Ground Truth', fontsize=13, fontweight='bold')
        axes[1].axis('off')
        plt.colorbar(im1, ax=axes[1], fraction=0.046)
        
        # Prediction
        im2 = axes[2].imshow(pred, cmap='tab20', interpolation='nearest')
        axes[2].set_title('Model Prediction', fontsize=13, fontweight='bold')
        axes[2].axis('off')
        plt.colorbar(im2, ax=axes[2], fraction=0.046)
        
        plt.suptitle(f'Image {image_idx} - Segmentation Comparison', fontsize=14, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.show()
        
        # Statistics
        print(f"\n📊 Image {image_idx} Statistics:")
        print(f"  Image shape: {img_np.shape}")
        print(f"  GT classes: {len(np.unique(gt))} unique values")
        print(f"  Pred classes: {len(np.unique(pred))} unique values")
        print(f"  GT range: [{gt.min()}, {gt.max()}]")
        print(f"  Pred range: [{pred.min()}, {pred.max()}]")

# Show predictions for different images
for idx in range(min(3, len(images))):
    print(f"\n{'='*60}")
    show_prediction(model, idx, images_tensor, labels_2d)

## 🎨 Change the image index in the cell below and re-run to view different predictions

In [ ]:
# ============ INTERACTIVE PREDICTION VIEWER ============

image_idx = 0  # ← Change this number (0 to {}) to view different images

print(f"Showing prediction for image {image_idx}...\n")
show_prediction(model, image_idx, images_tensor, labels_2d)